<div style="background-color: #0f3460; background: linear-gradient(135deg, #0f3460, #533483, #e94560); padding: 30px 30px 20px 30px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: #ffffff; margin: 0; font-size: 2em;">Lab 06: Building Custom Evaluators (Custom Graders)</h1>
  <p style="color: #cccccc; font-size: 1.1em; margin-top: 8px;">
    Microsoft Foundry Agent Observability
  </p>
</div>

## What You Will Learn

In [Lab 03a](lab-03a-tools.ipynb), we added **Function Tools** to the Contoso Travel agent, enabling it to call three tools: `search_flights`, `search_hotels`, and `search_car_rentals`. The agent selects the appropriate tool based on the customer's query.

In this lab, we will quantitatively evaluate the **accuracy of tool selection** using **Custom Graders**.

| Phase | Content |
|-------|---------|
| **Phase 1** | Run the agent with real tools and collect actual tool call data |
| **Phase 2** | Evaluate the collected data using 3 types of Custom Graders |

Custom Grader types:

| Grader Type | Name | What It Evaluates |
|-------------|------|-------------------|
| `python` | `correct_tool_called` | Whether the expected tool was selected (0.0 / 1.0) |
| `python` | `required_params_present` | Whether required parameters are present (partial credit) |
| `score_model` | `routing_quality` | Overall routing quality assessment via LLM-as-judge (1–5) |

---

## 1. Setup

---

In [ ]:
import os
import json
import time
import pandas as pd
from dotenv import load_dotenv
from azure.identity import AzureCliCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, FunctionTool, Tool
from openai.types.eval_create_params import DataSourceConfigCustom
from openai.types.responses.response_input_param import FunctionCallOutput

load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.path.abspath(".")), ".env"))

tenant_id = os.getenv("TENANT_ID")
endpoint = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
model_name = os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"]

credential = AzureCliCredential(tenant_id=tenant_id)
project_client = AIProjectClient(endpoint=endpoint, credential=credential)
openai_client = project_client.get_openai_client()

print("✅ Connected to Microsoft Foundry")
print(f"   Model: {model_name}")

---

## Phase 1: Collecting Real Tool Call Data

Following the same steps as Lab 03a, we create an agent with **Function Tools** to search real data, then run it against evaluation queries. We collect the `function_call` responses (tool names and parameters) returned by the agent, which will be used as evaluation data in Phase 2.

### 2. Loading Travel Data and Defining Tool Functions

---

In [ ]:
data_path = "../../data/contoso-travel"
flights_df = pd.read_csv(f"{data_path}/flights.csv")
hotels_df = pd.read_csv(f"{data_path}/hotels.csv")
cars_df = pd.read_csv(f"{data_path}/car_rentals.csv")

print(f"\u2708\ufe0f  Loaded {len(flights_df)} flights")
print(f"\U0001f3e8 Loaded {len(hotels_df)} hotels")
print(f"\U0001f697 Loaded {len(cars_df)} car rentals")


def search_flights(origin: str = None, destination: str = None, cabin_class: str = None, max_price: float = None) -> str:
    """Search for available flights based on criteria."""
    results = flights_df.copy()
    if origin:
        results = results[results["origin"].str.lower() == origin.lower()]
    if destination:
        results = results[results["destination"].str.lower() == destination.lower()]
    if cabin_class:
        results = results[results["cabin_class"].str.lower() == cabin_class.lower()]
    if max_price:
        results = results[results["price_usd"] <= max_price]
    if results.empty:
        return json.dumps({"message": "No flights found.", "results": []})
    return results.to_json(orient="records")


def search_hotels(city: str = None, min_stars: int = None, max_price: float = None, amenities: str = None) -> str:
    """Search for available hotels based on criteria."""
    results = hotels_df.copy()
    if city:
        results = results[results["city"].str.lower() == city.lower()]
    if min_stars:
        results = results[results["star_rating"] >= min_stars]
    if max_price:
        results = results[results["price_per_night_usd"] <= max_price]
    if amenities:
        results = results[results["amenities"].str.lower().str.contains(amenities.lower())]
    if results.empty:
        return json.dumps({"message": "No hotels found.", "results": []})
    return results.to_json(orient="records")


def search_car_rentals(city: str = None, car_type: str = None, max_price: float = None) -> str:
    """Search for available car rentals based on criteria."""
    results = cars_df.copy()
    if city:
        results = results[results["city"].str.lower() == city.lower()]
    if car_type:
        results = results[results["car_type"].str.lower() == car_type.lower()]
    if max_price:
        results = results[results["price_per_day_usd"] <= max_price]
    results = results[results["available"] == True]
    if results.empty:
        return json.dumps({"message": "No car rentals found.", "results": []})
    return results.to_json(orient="records")


print("\n\u2705 Defined 3 search functions")

In [ ]:
flight_tool = FunctionTool(
    name="search_flights",
    description="Search for available flights. Use when customer asks about flights or airfare.",
    parameters={
        "type": "object",
        "properties": {
            "origin":       {"type": "string", "description": "Departure city"},
            "destination":  {"type": "string", "description": "Arrival city"},
            "cabin_class":  {"type": "string", "enum": ["Economy", "Business", "First"]},
            "max_price":    {"type": "number", "description": "Max price in USD"},
        },
        "required": [],
        "additionalProperties": False,
    },
    strict=False,
)

hotel_tool = FunctionTool(
    name="search_hotels",
    description="Search for available hotels. Use when customer asks about hotels or accommodation.",
    parameters={
        "type": "object",
        "properties": {
            "city":      {"type": "string"},
            "min_stars": {"type": "integer"},
            "max_price": {"type": "number"},
            "amenities": {"type": "string", "description": "e.g. Pool, Spa, WiFi"},
        },
        "required": [],
        "additionalProperties": False,
    },
    strict=False,
)

car_rental_tool = FunctionTool(
    name="search_car_rentals",
    description="Search for car rentals. Use when customer asks about rental cars.",
    parameters={
        "type": "object",
        "properties": {
            "city":     {"type": "string"},
            "car_type": {"type": "string", "enum": ["Economy", "SUV", "Luxury", "Minivan"]},
            "max_price": {"type": "number"},
        },
        "required": [],
        "additionalProperties": False,
    },
    strict=False,
)

tools: list[Tool] = [flight_tool, hotel_tool, car_rental_tool]

TRAVEL_AGENT_INSTRUCTIONS = """You are a Contoso Travel agent with access to real travel data.
Use the appropriate tools to answer customer questions.
Always search before responding — do not guess."""

agent = project_client.agents.create_version(
    agent_name="contoso-travel-agent-eval",
    definition=PromptAgentDefinition(
        model=model_name,
        instructions=TRAVEL_AGENT_INSTRUCTIONS,
        tools=tools,
    ),
)

print(f"\u2705 Created agent with tools: {agent.name} (v{agent.version})")
print(f"   Registered tools: {[t.name for t in tools]}")

### 3. Defining Test Queries and Collecting Tool Call Data

For each query, we run the agent and collect the `function_call` items (tool name and parameters) returned. This becomes the evaluation data for Phase 2.

---

In [ ]:
test_queries = [
    {
        "query": "Are there any flights from Seattle to Paris in August?",
        "expected_tool": "search_flights",
        "expected_params": {"origin": "Seattle", "destination": "Paris"},
    },
    {
        "query": "Find me a hotel with a spa in Tokyo.",
        "expected_tool": "search_hotels",
        "expected_params": {"city": "Tokyo", "amenities": "Spa"},
    },
    {
        "query": "I need a cheap car rental in Cancun.",
        "expected_tool": "search_car_rentals",
        "expected_params": {"city": "Cancun"},
    },
    {
        "query": "Find me a business class flight to New York.",
        "expected_tool": "search_flights",
        "expected_params": {"destination": "New York", "cabin_class": "Business"},
    },
    {
        "query": "Find a 5-star hotel in Paris within a budget of $300.",
        "expected_tool": "search_hotels",
        "expected_params": {"city": "Paris", "min_stars": 5},
    },
    {
        "query": "I'm looking for an SUV rental in Los Angeles.",
        "expected_tool": "search_car_rentals",
        "expected_params": {"city": "Los Angeles", "car_type": "SUV"},
    },
]

# Run the agent for each query and collect function_call responses
collected_items = []

for q in test_queries:
    conversation = openai_client.conversations.create()
    response = openai_client.responses.create(
        conversation=conversation.id,
        extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
        input=q["query"],
    )

    for output_item in response.output:
        if output_item.type == "function_call":
            collected_items.append({
                "query":           q["query"],
                "actual_tool":     output_item.name,
                "actual_args":     output_item.arguments,   # JSON string
                "expected_tool":   q["expected_tool"],
                "expected_params": json.dumps(q["expected_params"], ensure_ascii=False),
            })
            print(f"\u2705 {output_item.name}({output_item.arguments[:60]}...)")
            break  # collect only the first function_call

    openai_client.conversations.delete(conversation_id=conversation.id)

print(f"\n\U0001f4cb Collected {len(collected_items)} tool call data items")

### 3.1 Saving and Uploading as a JSONL File

Save the collected data to `custom_evaluation_data.jsonl` and upload it to Foundry as a versioned dataset. The uploaded file can be referenced by `file_id` in evaluation runs.

---

In [ ]:
jsonl_path = f"{data_path}/custom_evaluation_data.jsonl"

# Save as JSONL: one flat JSON object per line (no {"item": {...}} wrapper needed)
with open(jsonl_path, "w", encoding="utf-8") as f:
    for ci in collected_items:
        f.write(json.dumps(ci, ensure_ascii=False) + "\n")

print(f"✅ Saved JSONL to: {jsonl_path}")
print(f"   Records: {len(collected_items)}")

# Upload as a Foundry Dataset
dataset_name    = "contoso-travel-tool-calls"
dataset_version = "1"

dataset = project_client.datasets.upload_file(
    name=dataset_name,
    version=dataset_version,
    file_path=jsonl_path,
)
data_id = dataset.id

print(f"\n✅ Uploaded dataset")
print(f"   Dataset ID: {data_id}")
print(f"   Name: {dataset_name} (v{dataset_version})")

---

## Phase 2: Evaluating with Custom Graders

We evaluate the collected tool call data (`actual_tool`, `actual_args`) using Custom Graders. All graders reference `item.*` fields (collected data), so no additional model calls are needed during evaluation.

### 4. Defining Custom Graders

#### 4.1 `python` Graders — `correct_tool_called` and `required_params_present`

The `grade(sample, item)` function runs in Foundry's sandbox. `item` contains the collected tool call data.

---

In [ ]:
# Python grader 1: whether the correct tool was selected
correct_tool_grader_source = r"""
def grade(sample, item) -> float:
    return 1.0 if item.get("actual_tool") == item.get("expected_tool") else 0.0
"""

# Python grader 2: whether required parameters are present (partial credit)
required_params_grader_source = r"""
import json

def grade(sample, item) -> float:
    try:
        actual   = json.loads(item.get("actual_args",     "{}"))
        expected = json.loads(item.get("expected_params", "{}"))
        if not expected:
            return 1.0
        matched = sum(1 for k in expected if k in actual)
        return matched / len(expected)
    except (json.JSONDecodeError, AttributeError):
        return 0.0
"""

print(correct_tool_grader_source)
print(required_params_grader_source)

#### 4.2 `score_model` Grader — `routing_quality`

This is an LLM-as-judge grader. It evaluates the appropriateness of the tool and parameters selected by the agent for the customer's query on a scale of 1 to 5. While `correct_tool_called` is binary (0/1), this grader can **evaluate subtle judgment errors in a graduated manner**.

---

In [ ]:
custom_data_source_config = DataSourceConfigCustom(
    type="custom",
    item_schema={
        "type": "object",
        "properties": {
            "query":           {"type": "string"},
            "actual_tool":     {"type": "string"},
            "actual_args":     {"type": "string"},
            "expected_tool":   {"type": "string"},
            "expected_params": {"type": "string"},
        },
        "required": ["query", "actual_tool", "actual_args", "expected_tool", "expected_params"],
    },
    include_sample_schema=False,  # grader uses only item.* fields
)

custom_testing_criteria = [
    # (1) Python grader: whether the correct tool was selected
    {
        "type": "python",
        "name": "correct_tool_called",
        "source": correct_tool_grader_source,
        "pass_threshold": 1.0,
    },
    # (2) Python grader: whether required parameters are present
    {
        "type": "python",
        "name": "required_params_present",
        "source": required_params_grader_source,
        "pass_threshold": 1.0,
    },
    # (3) Score model grader: overall routing quality assessment via LLM-as-judge (1-5)
    {
        "type": "score_model",
        "name": "routing_quality",
        "model": model_name,
        "input": [
            {
                "role": "system",
                "content": (
                    "You are a quality inspector for a travel AI system. "
                    "Evaluate the appropriateness of the tool and parameters "
                    "the agent selected for the customer's question on a scale of 1 to 5 (integer only). "
                    "Return only the number.\n"
                    "5 = Perfect, 4 = Almost appropriate, 3 = Average, 2 = Slightly inappropriate, 1 = Completely wrong"
                ),
            },
            {
                "role": "user",
                "content": (
                    "Customer question: {{item.query}}\n"
                    "Called tool: {{item.actual_tool}}\n"
                    "Parameters: {{item.actual_args}}"
                ),
            },
        ],
        "range": [1, 5],
        "pass_threshold": 4.0,
    },
]

custom_eval = openai_client.evals.create(
    name="Contoso Travel - Tool Call Custom Graders",
    data_source_config=custom_data_source_config,
    testing_criteria=custom_testing_criteria,
)

print(f"\u2705 Created custom evaluation (ID: {custom_eval.id})")
for c in custom_eval.testing_criteria:
    print(f"   \u2022 {c.name} ({c.type})")

### 5. Running the Custom Evaluation

We reference the uploaded dataset by `file_id` to run the evaluation. Since all graders reference only `item.*` fields, we use **Dataset evaluation** mode (`type: "jsonl"`). No additional agent calls are made.

---

In [ ]:
custom_data_source = {
    "type": "jsonl",
    "source": {
        "type": "file_id",
        "id": data_id,
    },
}

custom_run = openai_client.evals.runs.create(
    eval_id=custom_eval.id,
    name="Tool Call Grader Run - Contoso Travel",
    data_source=custom_data_source,
)

print(f"✅ Started evaluation run (ID: {custom_run.id})")
print(f"   Status: {custom_run.status}")

while custom_run.status not in ["completed", "failed"]:
    custom_run = openai_client.evals.runs.retrieve(
        run_id=custom_run.id, eval_id=custom_eval.id
    )
    print(f"   ⏳ Status: {custom_run.status}")
    time.sleep(5)

print(f"\n{'✅' if custom_run.status == 'completed' else '❌'} Evaluation {custom_run.status}!")

### 6. Interpreting Results

| Evaluator | Score Meaning | Red Flags |
|-----------|--------------|----------|
| `correct_tool_called`     | 1.0 = Correct / 0.0 = Incorrect        | 0.0 → Wrong tool selected |
| `required_params_present` | 0.0–1.0 (parameter match rate)         | 0.5 or below → Required parameters missing |
| `routing_quality`         | 1–5 (pass at 4 or above)              | 3 or below → Routing judgment is inappropriate |

---

In [ ]:
if custom_run.status == "completed":
    print(f"\U0001f6e0\ufe0f Custom Evaluation Results")
    print(f"   Result counts: {custom_run.result_counts}")

    output_items = list(
        openai_client.evals.runs.output_items.list(
            run_id=custom_run.id, eval_id=custom_eval.id
        )
    )

    print(f"\n{'='*70}")
    for item in output_items:
        q            = item.datasource_item.get("query", "N/A")
        actual_tool  = item.datasource_item.get("actual_tool", "N/A")
        print(f"\nQuery:  {q[:60]}")
        print(f"Tool:   {actual_tool}")
        if hasattr(item, "results") and item.results:
            for result in item.results:
                name   = getattr(result, "name", "N/A")
                score  = getattr(result, "score", "N/A")
                passed = getattr(result, "passed", "N/A")
                print(f"  {name:28s}: score={score}, passed={passed}")
    print(f"{'='*70}")
else:
    print("\u274c Evaluation failed. Check the Foundry portal for details.")

### 7. Viewing Results in the Foundry Portal

1. Go to the [Microsoft Foundry Portal](https://ai.azure.com)
2. Open your project
3. Click the **Evaluations** tab in the left navigation
4. Click on the **Contoso Travel - Tool Call Custom Graders** run
5. For each of the 3 custom graders, you can view per-query scores and reasoning (for LLM-as-judge)

---

## 8. Cleanup

---

In [ ]:
openai_client.evals.delete(eval_id=custom_eval.id)
print("✅ Deleted custom evaluation")

project_client.datasets.delete_version(name=dataset_name, version=dataset_version)
print("✅ Deleted dataset")

project_client.agents.delete(agent_name=agent.name)
print("✅ Deleted agent")

openai_client.close()
project_client.close()
credential.close()
print("✅ Closed clients")